# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. 5, (2, 3)
2. 1, 8
3. 99
4. 2 1
5. (2,3)
   0.25
   


In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [ ]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — че
рез маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [ ]:
np.set_printoptions(threshold=20) # чтобы NumPy сокращал длинные массивы

In [ ]:
# База

index = amount.argmax()

print(amount.max(), cust[index], city[index])

9999990.0 TEST Алматы


In [ ]:
print ("Сумма оплаченных заказов:", amount[paid].sum())
print ("Сумма неоплаченных заказов:", amount[~paid].sum())

Сумма оплаченных заказов: 101909360.0
Сумма неоплаченных заказов: 7974000.0


In [ ]:
mask = city == "Шымкент"
print (f"Количество заказов в Шымкенте: {mask.sum()}")
print (f"Доля заказов на Шымкент: {mask.sum() / len(city) * 100:.4f}%")

Количество заказов в Шымкенте: 804
Доля заказов на Шымкент: 14.7172%


## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [ ]:
# Стандарт

mask = (amount > 50000) & paid
total_sum = amount[mask].sum()
print ("Магазин недополучит :", total_sum * 0.15)

mask2 = (amount > 50000) & paid & (cust != "TEST")
total_sum2 = amount[mask2].sum()
print ("Без заказов клиента TEST: ", total_sum2 * 0.15)

print (f"Ответы различаются в {(total_sum * 0.15)/(total_sum2 * 0.15):.2f} раза")
print ("Ответы различаются в несколько раз потому что заказы где клиентом является TEST сильно увеличивают общую выручку")

Магазин недополучит : 5702422.5
Без заказов клиента TEST:  1202427.0
Ответы различаются в 4.74 раза
Ответы различаются в несколько раз потому что заказы где клиентом является TEST сильно увеличивают общую выручку


In [ ]:
mask = (days == 1) | (days == 2)
total = len(days)
print (total, len(days[mask]))

for i in np.unique(city):
  mask2 = ( (days == 1) | (days == 2) ) & (city == i)
  total2 = len(days[city == i])
  print (f"Доля заказов на {i}, доставленных за 1-2 дня: { (len(days[mask2]) / total2 ) * 100 :.2f}")

5463 3241
Доля заказов на Актобе, доставленных за 1-2 дня: 62.22
Доля заказов на Алматы, доставленных за 1-2 дня: 64.53
Доля заказов на Астана, доставленных за 1-2 дня: 63.53
Доля заказов на Атырау, доставленных за 1-2 дня: 64.38
Доля заказов на Караганда, доставленных за 1-2 дня: 67.84
Доля заказов на Шымкент, доставленных за 1-2 дня: 29.85


In [ ]:
# Чтобы правильно посчитать средний чек нужно учитывать только оплаченные заказы и также исключить тестовые заказы где клиент - TEST

mask = (paid == True) & (cust != "TEST")
avg = amount[mask].mean()
print ("Средний чек:", round (avg, 2))

Средний чек: 14672.39


## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [ ]:
# Челлендж
